# IA45 extreme optical outlier audit
## Augmented 2025 support · true NCA footprint

This notebook searches the field-linked IA45 dataset for observations that are unusually extreme in optical feature space **before** those observations are used for PCA, class-support geometry, or supervised classification.

The audit deliberately separates three concepts:

1. **Global optical anomaly**  
   An observation is unusual relative to the entire empirical optical dataset.

2. **Multivariate leverage / influence**  
   An observation materially changes the covariance/PCA geometry used to represent the dataset.

3. **Class-conditional mismatch**  
   An observation may be optically plausible overall but unusual for its vegetation label.

Only the first two are candidates for technical/QC exclusion. Class-conditional mismatch is primarily a scientific diagnostic and is **not** an automatic exclusion criterion.

### Dataset

The audit uses the same augmented support intended for the PCA diagnostic:

- contemporaneous 2016 / 2018 / 2025 vegetation-optical observations;
- 2026 vegetation observations paired diagnostically with 2025 optics;
- true NCA footprint only;
- K5 and K8 labels attached to the same observations.

The pseudo-2026 observations remain explicitly marked:

\[
V_{2026} \leftrightarrow Z_{2025}.
\]

### Global diagnostics

For every observation the notebook calculates:

\[
M_i = \max_j |z_{ij}|
\]

\[
R_i = \sqrt{\frac{1}{45}\sum_j z_{ij}^2}
\]

plus counts of features exceeding several absolute-z thresholds, PCA-space extremeness, local k-nearest-neighbor isolation, and shrinkage-Mahalanobis distance.

The notebook also asks whether a small set of observations disproportionately changes the PCA basis using exact leave-one-out PCA refits for the strongest candidates.

### Important interpretation rule

This notebook **flags observations for review; it does not automatically delete them**.

A defensible exclusion should combine statistical evidence with a technical reason such as a pathological harmonic fit, nodata/edge contamination, geolocation error, or another identifiable optical-data problem.

In [ ]:
# =============================================================================
# 0. IMPORTS / PATHS / SETTINGS
# =============================================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors


pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 320)


# -----------------------------------------------------------------------------
# EXISTING CONTEMPORANEOUS K5 MODELING DATA
# -----------------------------------------------------------------------------

K5_BASELINE_DATASET = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_IA45_K5"
    r"\FieldVeg_K5_AnnualZScore_IA45_modeling_dataset.csv"
)


# -----------------------------------------------------------------------------
# RESOLVED 2026 VEGETATION -> 2025 OPTICAL SUPPORT
# -----------------------------------------------------------------------------

PSEUDO_2026_DATASET = Path(
    r"C:\NCA_DATA\Analysis\Diagnostics"
    r"\K5_2026Veg_as2025Optics_RESOLVED_v1"
    r"\Tables"
    r"\DIAGNOSTIC_2026veg_sampled_at_2025_IA45_z.csv"
)


# -----------------------------------------------------------------------------
# K8 VEGETATION ASSIGNMENTS
# -----------------------------------------------------------------------------

K8_ASSIGNMENTS = Path(
    r"C:\NCA_DATA\Vegetation Data\Clustering_Veg\Derived"
    r"\NCA_FCM_K8_m1p33_parent_assignments.csv"
)


# -----------------------------------------------------------------------------
# TRUE NCA FOOTPRINT
# -----------------------------------------------------------------------------

TRUE_NCA_BOUNDARY = Path(
    r"C:\NCA_DATA\templates\SRBOP_NCA_extent_26911.shp"
)


# -----------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------

OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\Diagnostics"
    r"\IA45_Optical_Outlier_Audit_Augmented2025_TrueNCA_v1"
)

TABLE_DIR = OUTPUT_DIR / "Tables"
FIG_DIR = OUTPUT_DIR / "Figures"

ALLOW_OVERWRITE = False


for directory in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


existing_outputs = [
    p
    for p in OUTPUT_DIR.rglob("*")
    if p.is_file()
]


if existing_outputs and not ALLOW_OVERWRITE:
    raise FileExistsError(
        "Outlier-audit output directory already contains files.\n"
        "Nothing will be overwritten.\n\n"
        f"{OUTPUT_DIR}"
    )


# -----------------------------------------------------------------------------
# LABELS
# -----------------------------------------------------------------------------

K5_LABELS = {
    1: "PBG",
    2: "ARTR",
    3: "Non-ARTR shrub",
    4: "Exotic forb",
    5: "EAG",
}

K8_LABELS = {
    1: "P1",
    2: "P2",
    3: "P3",
    4: "P4",
    5: "P5",
    6: "P6",
    7: "P7",
    8: "P8",
}


# -----------------------------------------------------------------------------
# OUTLIER DIAGNOSTIC SETTINGS
# -----------------------------------------------------------------------------

PCA_LOCAL_K = 10

KNN_K_SMALL = 5
KNN_K_LARGE = 10

EMPIRICAL_FLAG_QUANTILE = 0.99

# Hard, transparent annual-z review thresholds.
ABS_Z_REVIEW = 6.0
ABS_Z_HIGH = 8.0
ABS_Z_CRITICAL = 10.0

# Exact leave-one-out PCA influence is run only for the strongest candidates.
INFLUENCE_TOP_N = 30
INFLUENCE_PCS = 10

# How many rows to show / label in figures and audit tables.
TOP_N_DISPLAY = 30
TOP_FEATURES_PER_CANDIDATE = 10


print("IA45 EXTREME OPTICAL OUTLIER AUDIT")
print("Baseline:", K5_BASELINE_DATASET)
print("Pseudo-2026 support:", PSEUDO_2026_DATASET)
print("K8:", K8_ASSIGNMENTS)
print("True NCA:", TRUE_NCA_BOUNDARY)
print("Output:", OUTPUT_DIR)

In [ ]:
# =============================================================================
# 1. HELPERS
# =============================================================================

def clean_plot_id(value):

    if pd.isna(value):
        return None

    s = str(value).strip()

    if (
        s == ""
        or s.lower() == "nan"
    ):
        return None

    if re.fullmatch(
        r"-?\d+\.0",
        s,
    ):
        s = s[:-2]

    return s


def structural_plot_key(value):

    s = clean_plot_id(
        value
    )

    if s is None:
        return None

    s = s.lower()

    s = re.sub(
        r"^(?:\d{8}|\d{6})[_\-\s]+",
        "",
        s,
    )

    s = re.sub(
        r"[^a-z0-9]+",
        "",
        s,
    )

    return (
        s
        if s
        else None
    )


def write_csv_safe(
    df,
    path,
):

    path = Path(
        path
    )

    if (
        path.exists()
        and not ALLOW_OVERWRITE
    ):
        raise FileExistsError(
            path
        )

    df.to_csv(
        path,
        index=False,
    )

    print(
        "Wrote:",
        path,
    )


def empirical_percentile(
    values,
):

    return pd.Series(
        values
    ).rank(
        method="average",
        pct=True,
    ).to_numpy()


def robust_standardize_columns(
    X,
):

    median = np.median(
        X,
        axis=0,
    )

    mad = np.median(
        np.abs(
            X
            - median
        ),
        axis=0,
    )

    # 1.4826 * MAD approximates SD for a Gaussian distribution.
    scale = 1.4826 * mad

    scale = np.where(
        scale <= 0,
        1.0,
        scale,
    )

    Z = (
        X
        - median
    ) / scale

    return (
        Z,
        median,
        scale,
    )


def squared_mahalanobis(
    X,
    location,
    precision,
):

    delta = (
        X
        - location
    )

    return np.einsum(
        "ij,jk,ik->i",
        delta,
        precision,
        delta,
    )

## 2. Assemble the augmented common K5/K8 dataset

No previously identified outlier is removed here. The point of this notebook is to discover the outliers from the complete augmented dataset.

In [ ]:
# =============================================================================
# 2. LOAD BASELINE + PSEUDO-2026 SUPPORT
# =============================================================================

for path in [
    K5_BASELINE_DATASET,
    PSEUDO_2026_DATASET,
    K8_ASSIGNMENTS,
    TRUE_NCA_BOUNDARY,
]:

    if not path.exists():
        raise FileNotFoundError(
            path
        )


baseline = pd.read_csv(
    K5_BASELINE_DATASET,
    low_memory=False,
).drop(
    columns=[
        "Unnamed: 0",
    ],
    errors="ignore",
)


pseudo = pd.read_csv(
    PSEUDO_2026_DATASET,
    low_memory=False,
).drop(
    columns=[
        "Unnamed: 0",
    ],
    errors="ignore",
)


# -----------------------------------------------------------------------------
# IDENTIFY THE 45 ANNUAL-Z FEATURES FROM THE BASELINE TABLE
# -----------------------------------------------------------------------------

Z_COLS = [
    c
    for c in baseline.columns
    if str(
        c
    ).startswith(
        "z__"
    )
]


if len(
    Z_COLS
) != 45:
    raise ValueError(
        f"Expected 45 annual-z IA45 predictors; found {len(Z_COLS)}."
    )


FEATURES = [
    c.replace(
        "z__",
        "",
        1,
    )
    for c in Z_COLS
]


missing_pseudo_z = [
    c
    for c in Z_COLS
    if c not in pseudo.columns
]


if missing_pseudo_z:
    raise KeyError(
        f"Pseudo-2026 table is missing {len(missing_pseudo_z)} z-score predictors."
    )


# -----------------------------------------------------------------------------
# BASELINE KEYS
# -----------------------------------------------------------------------------

baseline[
    "PlotID_model"
] = baseline[
    "PlotID"
].map(
    clean_plot_id
)

baseline[
    "_plot_key"
] = baseline[
    "PlotID_model"
].map(
    structural_plot_key
)

baseline[
    "veg_year_original"
] = pd.to_numeric(
    baseline[
        "Year"
    ],
    errors="raise",
).astype(int)

baseline[
    "optical_year"
] = baseline[
    "veg_year_original"
]

baseline[
    "source"
] = "baseline_contemporaneous"


# -----------------------------------------------------------------------------
# PSEUDO-2026 KEYS
# -----------------------------------------------------------------------------

pseudo_id_candidates = [
    c
    for c in [
        "Plot",
        "PlotID",
        "coordinate_master_PlotID",
    ]
    if c in pseudo.columns
]


if not pseudo_id_candidates:
    raise KeyError(
        "Pseudo-2026 table contains no usable plot identifier."
    )


PSEUDO_ID_COL = pseudo_id_candidates[
    0
]


pseudo[
    "PlotID_model"
] = pseudo[
    PSEUDO_ID_COL
].map(
    clean_plot_id
)

pseudo[
    "_plot_key"
] = pseudo[
    "PlotID_model"
].map(
    structural_plot_key
)


if "veg_year" in pseudo.columns:

    pseudo[
        "veg_year_original"
    ] = pd.to_numeric(
        pseudo[
            "veg_year"
        ],
        errors="raise",
    ).astype(int)

else:

    pseudo[
        "veg_year_original"
    ] = pd.to_numeric(
        pseudo[
            "Year"
        ],
        errors="raise",
    ).astype(int)


pseudo[
    "optical_year"
] = 2025

pseudo[
    "source"
] = "pseudo_2026_as_2025"


# -----------------------------------------------------------------------------
# COMMON SCHEMA
# -----------------------------------------------------------------------------

required_common = [
    "PlotID_model",
    "_plot_key",
    "veg_year_original",
    "optical_year",
    "source",
    "Easting",
    "Northing",
    "parent_cluster_k5",
] + Z_COLS


baseline_common = baseline[
    required_common
].copy()

pseudo_common = pseudo[
    required_common
].copy()


combined = pd.concat(
    [
        baseline_common,
        pseudo_common,
    ],
    ignore_index=True,
)


for col in [
    "Easting",
    "Northing",
]:

    combined[
        col
    ] = pd.to_numeric(
        combined[
            col
        ],
        errors="coerce",
    )


combined[
    "parent_cluster_k5"
] = pd.to_numeric(
    combined[
        "parent_cluster_k5"
    ],
    errors="coerce",
)


print(
    "Combined observations:",
    len(
        combined
    )
)

display(
    combined[
        "source"
    ].value_counts().rename(
        "n"
    ).to_frame()
)

In [ ]:
# =============================================================================
# 3. ATTACH K8 LABELS
#
# Repeated source rows are acceptable if the Plot-Year maps to the same K8 label.
# Conflicting labels hard-stop.
# =============================================================================

k8 = pd.read_csv(
    K8_ASSIGNMENTS,
    low_memory=False,
).drop(
    columns=[
        "Unnamed: 0",
    ],
    errors="ignore",
)


if "Plot" in k8.columns:

    K8_PLOT_COL = "Plot"

elif "PlotID" in k8.columns:

    K8_PLOT_COL = "PlotID"

else:

    raise KeyError(
        "K8 assignment table contains neither Plot nor PlotID."
    )


if "parent_cluster" in k8.columns:

    K8_LABEL_COL = "parent_cluster"

elif "parent_cluster_k8" in k8.columns:

    K8_LABEL_COL = "parent_cluster_k8"

else:

    raise KeyError(
        "Could not identify K8 parent-cluster column."
    )


k8[
    "Year"
] = pd.to_numeric(
    k8[
        "Year"
    ],
    errors="coerce",
)

k8[
    "_plot_key"
] = k8[
    K8_PLOT_COL
].map(
    structural_plot_key
)


k8_small = (
    k8[
        [
            "_plot_key",
            "Year",
            K8_LABEL_COL,
        ]
    ]
    .rename(
        columns={
            "Year": "veg_year_original",
            K8_LABEL_COL: "parent_cluster_k8",
        }
    )
    .copy()
)


k8_small[
    "parent_cluster_k8"
] = pd.to_numeric(
    k8_small[
        "parent_cluster_k8"
    ],
    errors="coerce",
)


label_counts = (
    k8_small.groupby(
        [
            "_plot_key",
            "veg_year_original",
        ]
    )[
        "parent_cluster_k8"
    ]
    .nunique(
        dropna=True
    )
)


conflicting = label_counts[
    label_counts
    > 1
]


if len(
    conflicting
) > 0:

    conflict_keys = (
        conflicting
        .reset_index()[
            [
                "_plot_key",
                "veg_year_original",
            ]
        ]
    )

    display(
        k8_small.merge(
            conflict_keys,
            on=[
                "_plot_key",
                "veg_year_original",
            ],
            how="inner",
        ).sort_values(
            [
                "_plot_key",
                "veg_year_original",
                "parent_cluster_k8",
            ]
        )
    )

    raise ValueError(
        "Conflicting K8 labels exist for at least one Plot-Year."
    )


k8_lookup = (
    k8_small
    .dropna(
        subset=[
            "_plot_key",
            "veg_year_original",
            "parent_cluster_k8",
        ]
    )
    .drop_duplicates(
        subset=[
            "_plot_key",
            "veg_year_original",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


k8_lookup[
    "parent_cluster_k8"
] = k8_lookup[
    "parent_cluster_k8"
].astype(int)


combined = combined.merge(
    k8_lookup,
    on=[
        "_plot_key",
        "veg_year_original",
    ],
    how="left",
    validate="many_to_one",
)


missing_k8 = combined[
    "parent_cluster_k8"
].isna()


print(
    "Rows missing K8:",
    int(
        missing_k8.sum()
    ),
    "/",
    len(
        combined
    ),
)


if missing_k8.mean() > 0.02:

    display(
        combined.loc[
            missing_k8,
            [
                "PlotID_model",
                "_plot_key",
                "veg_year_original",
                "source",
            ],
        ].head(
            50
        )
    )

    raise ValueError(
        "More than 2% of augmented observations failed the K8 join."
    )


combined = (
    combined.loc[
        ~missing_k8
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


combined[
    "parent_cluster_k8"
] = combined[
    "parent_cluster_k8"
].astype(int)

In [ ]:
# =============================================================================
# 4. TRUE NCA FOOTPRINT FILTER
# =============================================================================

nca = gpd.read_file(
    TRUE_NCA_BOUNDARY
)


if nca.crs is None:
    raise ValueError(
        "True NCA boundary has no CRS metadata."
    )


points = gpd.GeoDataFrame(
    combined.copy(),
    geometry=gpd.points_from_xy(
        combined[
            "Easting"
        ],
        combined[
            "Northing"
        ],
    ),
    crs="EPSG:26911",
)


if points.crs != nca.crs:

    nca = nca.to_crs(
        points.crs
    )


nca_union = nca.geometry.unary_union


inside = points.geometry.intersects(
    nca_union
)


analysis_df = (
    combined.loc[
        inside.to_numpy()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


finite_mask = np.isfinite(
    analysis_df[
        Z_COLS
    ].to_numpy(
        dtype=float
    )
).all(
    axis=1
)


if not finite_mask.all():

    bad = analysis_df.loc[
        ~finite_mask,
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "optical_year",
            "source",
        ],
    ]

    display(
        bad
    )

    raise ValueError(
        "Non-finite annual-z IA45 values exist in the true-NCA dataset."
    )


analysis_df[
    "parent_cluster_k5"
] = analysis_df[
    "parent_cluster_k5"
].astype(int)


X_z = analysis_df[
    Z_COLS
].to_numpy(
    dtype=np.float64
)


print(
    "True-NCA observations:",
    len(
        analysis_df
    )
)

print(
    "IA45 shape:",
    X_z.shape
)


display(
    analysis_df.groupby(
        [
            "source",
            "optical_year",
        ]
    ).size().rename(
        "n"
    ).to_frame()
)

# Part I — transparent row-wise extremeness

Because the predictors are already annual landscape z-scores, the first diagnostics are intentionally simple and interpretable.

A single very large \(|z|\) can identify an unstable harmonic parameter. Moderate extremeness across many variables instead produces a large RMS/vector norm.

In [ ]:
# =============================================================================
# 5. UNIVARIATE + WHOLE-VECTOR EXTREMENESS
# =============================================================================

abs_z = np.abs(
    X_z
)


analysis_df[
    "max_abs_z"
] = abs_z.max(
    axis=1
)

analysis_df[
    "mean_abs_z"
] = abs_z.mean(
    axis=1
)

analysis_df[
    "median_abs_z"
] = np.median(
    abs_z,
    axis=1,
)

analysis_df[
    "rms_z"
] = np.sqrt(
    np.mean(
        X_z ** 2,
        axis=1,
    )
)

analysis_df[
    "l2_z"
] = np.linalg.norm(
    X_z,
    axis=1,
)


for threshold in [
    4,
    5,
    6,
    8,
    10,
]:

    analysis_df[
        f"n_abs_z_gt_{threshold}"
    ] = (
        abs_z
        > threshold
    ).sum(
        axis=1
    )


max_feature_idx = np.argmax(
    abs_z,
    axis=1,
)


analysis_df[
    "most_extreme_feature"
] = [
    FEATURES[
        j
    ]
    for j in max_feature_idx
]


analysis_df[
    "most_extreme_feature_z"
] = X_z[
    np.arange(
        len(
            X_z
        )
    ),
    max_feature_idx,
]


analysis_df[
    "max_abs_z_percentile"
] = empirical_percentile(
    analysis_df[
        "max_abs_z"
    ]
)

analysis_df[
    "rms_z_percentile"
] = empirical_percentile(
    analysis_df[
        "rms_z"
    ]
)


display(
    analysis_df.sort_values(
        "max_abs_z",
        ascending=False,
    )[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "optical_year",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "max_abs_z",
            "rms_z",
            "n_abs_z_gt_6",
            "n_abs_z_gt_8",
            "n_abs_z_gt_10",
            "most_extreme_feature",
            "most_extreme_feature_z",
        ]
    ].head(
        TOP_N_DISPLAY
    ).round(
        4
    )
)

In [ ]:
# =============================================================================
# 6. FEATURE-LEVEL EXTREME-VALUE AUDIT
#
# This identifies parameters that repeatedly generate pathological tails.
# =============================================================================

feature_rows = []


for j, feature in enumerate(
    FEATURES
):

    values = X_z[
        :,
        j
    ]

    av = np.abs(
        values
    )


    feature_rows.append(
        {
            "feature": feature,
            "n": len(
                values
            ),
            "median_abs_z": float(
                np.median(
                    av
                )
            ),
            "p95_abs_z": float(
                np.quantile(
                    av,
                    0.95,
                )
            ),
            "p99_abs_z": float(
                np.quantile(
                    av,
                    0.99,
                )
            ),
            "max_abs_z": float(
                av.max()
            ),
            "n_abs_gt_4": int(
                np.sum(
                    av > 4
                )
            ),
            "n_abs_gt_5": int(
                np.sum(
                    av > 5
                )
            ),
            "n_abs_gt_6": int(
                np.sum(
                    av > 6
                )
            ),
            "n_abs_gt_8": int(
                np.sum(
                    av > 8
                )
            ),
            "n_abs_gt_10": int(
                np.sum(
                    av > 10
                )
            ),
        }
    )


feature_extreme_df = pd.DataFrame(
    feature_rows
).sort_values(
    [
        "max_abs_z",
        "p99_abs_z",
    ],
    ascending=False,
)


write_csv_safe(
    feature_extreme_df,
    TABLE_DIR
    / "DIAGNOSTIC_feature_extreme_tail_summary.csv",
)


display(
    feature_extreme_df.head(
        20
    ).round(
        4
    )
)


fig, ax = plt.subplots(
    figsize=(
        10,
        6,
    )
)


top_features_plot = feature_extreme_df.head(
    15
).iloc[
    ::-1
]


ax.barh(
    top_features_plot[
        "feature"
    ],
    top_features_plot[
        "max_abs_z"
    ],
)


ax.set_xlabel(
    "Maximum absolute annual-z value"
)

ax.set_ylabel(
    "IA45 feature"
)

ax.set_title(
    "IA45 parameters with the most extreme observed tails"
)

ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


fig_path = (
    FIG_DIR
    / "feature_max_abs_z_top15.png"
)


fig.savefig(
    fig_path,
    dpi=250,
    bbox_inches="tight",
)


plt.show()
plt.close(
    fig
)

# Part II — multivariate optical isolation

The next diagnostics ask whether an observation is unusual as a **45-dimensional optical vector**, even when no individual feature is catastrophic.

To reduce redundancy for local-neighbor distances, a 10-PC basis is fitted to a preliminary **core** subset that excludes only transparent gross-tail observations. Every observation, including the excluded gross-tail observations, is then projected into that core basis and scored.

In [ ]:
# =============================================================================
# 7. CORE PCA10 FOR LOCAL GEOMETRY
# =============================================================================

rms_q995 = np.quantile(
    analysis_df[
        "rms_z"
    ],
    0.995,
)


core_mask = (
    analysis_df[
        "max_abs_z"
    ].lt(
        ABS_Z_HIGH
    )
    & analysis_df[
        "rms_z"
    ].lt(
        rms_q995
    )
).to_numpy()


print(
    "Core PCA fitting rows:",
    int(
        core_mask.sum()
    ),
    "/",
    len(
        analysis_df
    ),
)


if core_mask.sum() < 100:
    raise ValueError(
        "Unexpectedly few observations remain for core PCA."
    )


core_pca = PCA(
    n_components=PCA_LOCAL_K,
    svd_solver="full",
)


core_pca.fit(
    X_z[
        core_mask
    ]
)


X_core_pc = core_pca.transform(
    X_z
)


analysis_df[
    "core_pca10_explained_variance"
] = float(
    core_pca.explained_variance_ratio_.sum()
)


# Robust distance in the core-PC representation.
core_pc_robust_z, _, _ = robust_standardize_columns(
    X_core_pc
)


analysis_df[
    "core_pc10_robust_radius"
] = np.linalg.norm(
    core_pc_robust_z,
    axis=1,
)


analysis_df[
    "core_pc10_robust_radius_percentile"
] = empirical_percentile(
    analysis_df[
        "core_pc10_robust_radius"
    ]
)


print(
    "Core PCA10 cumulative explained variance:",
    round(
        float(
            core_pca.explained_variance_ratio_.sum()
        ),
        4,
    )
)

In [ ]:
# =============================================================================
# 8. kNN LOCAL ISOLATION IN CORE PCA10 SPACE
# =============================================================================

nn = NearestNeighbors(
    n_neighbors=KNN_K_LARGE + 1,
    metric="euclidean",
)


nn.fit(
    X_core_pc
)


distances, neighbor_idx = nn.kneighbors(
    X_core_pc
)


# First neighbor is the observation itself at distance 0.
neighbor_distances = distances[
    :,
    1:
]


analysis_df[
    "knn5_mean_distance"
] = neighbor_distances[
    :,
    :KNN_K_SMALL
].mean(
    axis=1
)

analysis_df[
    "knn10_mean_distance"
] = neighbor_distances[
    :,
    :KNN_K_LARGE
].mean(
    axis=1
)

analysis_df[
    "knn10_distance_percentile"
] = empirical_percentile(
    analysis_df[
        "knn10_mean_distance"
    ]
)


display(
    analysis_df.sort_values(
        "knn10_mean_distance",
        ascending=False,
    )[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "max_abs_z",
            "rms_z",
            "core_pc10_robust_radius",
            "knn10_mean_distance",
        ]
    ].head(
        TOP_N_DISPLAY
    ).round(
        4
    )
)

In [ ]:
# =============================================================================
# 9. GLOBAL LEDOIT-WOLF MAHALANOBIS DISTANCE
#
# This is an additional multivariate diagnostic, not an automatic deletion rule.
# =============================================================================

lw = LedoitWolf(
    assume_centered=False,
)


lw.fit(
    X_z
)


global_md2 = squared_mahalanobis(
    X_z,
    lw.location_,
    lw.precision_,
)


analysis_df[
    "global_lw_mahalanobis_sq"
] = global_md2

analysis_df[
    "global_lw_mahal_percentile"
] = empirical_percentile(
    global_md2
)


display(
    analysis_df.sort_values(
        "global_lw_mahalanobis_sq",
        ascending=False,
    )[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "max_abs_z",
            "rms_z",
            "knn10_mean_distance",
            "global_lw_mahalanobis_sq",
        ]
    ].head(
        TOP_N_DISPLAY
    ).round(
        4
    )
)

# Part III — class-conditional optical mismatch

These diagnostics identify observations that are unusual **for their vegetation label**.

They are intentionally kept separate from the global outlier consensus. A legitimate ecological transition, unusual vegetation expression, disturbance history, or the deliberate 2026→2025 temporal mismatch could all produce a high class-conditional score without constituting bad optical data.

In [ ]:
# =============================================================================
# 10. SAME-CLASS kNN DISTANCES — K5 AND K8
# =============================================================================

def same_class_knn_mean_distance(
    X,
    y,
    k=5,
):

    out = np.full(
        len(
            y
        ),
        np.nan,
        dtype=float,
    )


    for cls in np.unique(
        y
    ):

        idx = np.where(
            y
            == cls
        )[
            0
        ]


        if len(
            idx
        ) < 3:
            continue


        k_use = min(
            k,
            len(
                idx
            )
            - 1,
        )


        nn_cls = NearestNeighbors(
            n_neighbors=k_use + 1,
            metric="euclidean",
        )


        nn_cls.fit(
            X[
                idx
            ]
        )


        d, _ = nn_cls.kneighbors(
            X[
                idx
            ]
        )


        out[
            idx
        ] = d[
            :,
            1:
        ].mean(
            axis=1
        )


    return out


y5 = analysis_df[
    "parent_cluster_k5"
].to_numpy(
    dtype=int
)

y8 = analysis_df[
    "parent_cluster_k8"
].to_numpy(
    dtype=int
)


analysis_df[
    "k5_sameclass_knn5"
] = same_class_knn_mean_distance(
    X_core_pc,
    y5,
    k=5,
)

analysis_df[
    "k8_sameclass_knn5"
] = same_class_knn_mean_distance(
    X_core_pc,
    y8,
    k=5,
)


analysis_df[
    "k5_sameclass_knn_percentile"
] = empirical_percentile(
    analysis_df[
        "k5_sameclass_knn5"
    ]
)

analysis_df[
    "k8_sameclass_knn_percentile"
] = empirical_percentile(
    analysis_df[
        "k8_sameclass_knn5"
    ]
)


print(
    "Most isolated observations relative to their K5 class:"
)

display(
    analysis_df.sort_values(
        "k5_sameclass_knn5",
        ascending=False,
    )[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "max_abs_z",
            "knn10_mean_distance",
            "k5_sameclass_knn5",
        ]
    ].head(
        TOP_N_DISPLAY
    ).round(
        4
    )
)


print(
    "\nMost isolated observations relative to their K8 class:"
)

display(
    analysis_df.sort_values(
        "k8_sameclass_knn5",
        ascending=False,
    )[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "max_abs_z",
            "knn10_mean_distance",
            "k8_sameclass_knn5",
        ]
    ].head(
        TOP_N_DISPLAY
    ).round(
        4
    )
)

# Part IV — consensus review priority

The global consensus deliberately combines diagnostics with different geometry:

- maximum absolute annual-z;
- RMS annual-z;
- robust distance in core PCA10;
- local PCA10 kNN distance;
- global Ledoit-Wolf Mahalanobis distance.

An observation is not considered technically bad merely because it scores high on one diagnostic.

The priority labels are **review categories**, not deletion rules.

In [ ]:
# =============================================================================
# 11. GLOBAL CONSENSUS FLAGS
# =============================================================================

GLOBAL_PERCENTILE_METRICS = [
    "max_abs_z_percentile",
    "rms_z_percentile",
    "core_pc10_robust_radius_percentile",
    "knn10_distance_percentile",
    "global_lw_mahal_percentile",
]


for col in GLOBAL_PERCENTILE_METRICS:

    analysis_df[
        f"flag__{col}"
    ] = analysis_df[
        col
    ].ge(
        EMPIRICAL_FLAG_QUANTILE
    )


flag_cols = [
    f"flag__{col}"
    for col in GLOBAL_PERCENTILE_METRICS
]


analysis_df[
    "global_consensus_count"
] = analysis_df[
    flag_cols
].sum(
    axis=1
)


analysis_df[
    "hard_flag_abs_z_ge_6"
] = analysis_df[
    "max_abs_z"
].ge(
    ABS_Z_REVIEW
)

analysis_df[
    "hard_flag_abs_z_ge_8"
] = analysis_df[
    "max_abs_z"
].ge(
    ABS_Z_HIGH
)

analysis_df[
    "hard_flag_abs_z_ge_10"
] = analysis_df[
    "max_abs_z"
].ge(
    ABS_Z_CRITICAL
)


def assign_priority(
    row,
):

    if (
        row[
            "hard_flag_abs_z_ge_10"
        ]
        or row[
            "global_consensus_count"
        ] >= 4
    ):
        return "CRITICAL"

    if (
        row[
            "hard_flag_abs_z_ge_8"
        ]
        or row[
            "global_consensus_count"
        ] >= 3
    ):
        return "HIGH"

    if (
        row[
            "hard_flag_abs_z_ge_6"
        ]
        or row[
            "global_consensus_count"
        ] >= 2
    ):
        return "REVIEW"

    if row[
        "global_consensus_count"
    ] >= 1:
        return "WATCH"

    return "NORMAL"


analysis_df[
    "review_priority"
] = analysis_df.apply(
    assign_priority,
    axis=1,
)


priority_order = {
    "CRITICAL": 0,
    "HIGH": 1,
    "REVIEW": 2,
    "WATCH": 3,
    "NORMAL": 4,
}


analysis_df[
    "_priority_order"
] = analysis_df[
    "review_priority"
].map(
    priority_order
)


audit_sorted = analysis_df.sort_values(
    [
        "_priority_order",
        "global_consensus_count",
        "max_abs_z",
        "knn10_mean_distance",
    ],
    ascending=[
        True,
        False,
        False,
        False,
    ],
).copy()


display(
    audit_sorted[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "optical_year",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "review_priority",
            "global_consensus_count",
            "max_abs_z",
            "rms_z",
            "core_pc10_robust_radius",
            "knn10_mean_distance",
            "global_lw_mahalanobis_sq",
            "k5_sameclass_knn5",
            "k8_sameclass_knn5",
            "most_extreme_feature",
            "most_extreme_feature_z",
        ]
    ].head(
        50
    ).round(
        4
    )
)


print(
    "\nReview-priority counts:"
)

display(
    analysis_df[
        "review_priority"
    ].value_counts().rename(
        "n"
    ).to_frame()
)

In [ ]:
# =============================================================================
# 12. SOURCE / YEAR / CLASS SUMMARY
# =============================================================================

summary_source = (
    analysis_df.groupby(
        [
            "source",
            "optical_year",
        ],
        as_index=False,
    )
    .agg(
        n=(
            "_plot_key",
            "size",
        ),
        median_max_abs_z=(
            "max_abs_z",
            "median",
        ),
        max_max_abs_z=(
            "max_abs_z",
            "max",
        ),
        median_knn10=(
            "knn10_mean_distance",
            "median",
        ),
        n_critical=(
            "review_priority",
            lambda x: int(
                np.sum(
                    x == "CRITICAL"
                )
            ),
        ),
        n_high=(
            "review_priority",
            lambda x: int(
                np.sum(
                    x == "HIGH"
                )
            ),
        ),
        n_review=(
            "review_priority",
            lambda x: int(
                np.sum(
                    x == "REVIEW"
                )
            ),
        ),
    )
)


summary_source[
    "fraction_critical_or_high"
] = (
    summary_source[
        "n_critical"
    ]
    + summary_source[
        "n_high"
    ]
) / summary_source[
    "n"
]


write_csv_safe(
    summary_source,
    TABLE_DIR
    / "DIAGNOSTIC_outlier_summary_by_source_year.csv",
)


display(
    summary_source.round(
        4
    )
)


summary_k5 = (
    analysis_df.groupby(
        "parent_cluster_k5",
        as_index=False,
    )
    .agg(
        n=(
            "_plot_key",
            "size",
        ),
        median_max_abs_z=(
            "max_abs_z",
            "median",
        ),
        median_knn10=(
            "knn10_mean_distance",
            "median",
        ),
        median_sameclass_knn=(
            "k5_sameclass_knn5",
            "median",
        ),
        n_critical_high=(
            "review_priority",
            lambda x: int(
                np.sum(
                    x.isin(
                        [
                            "CRITICAL",
                            "HIGH",
                        ]
                    )
                )
            ),
        ),
    )
)


summary_k5[
    "k5_label"
] = summary_k5[
    "parent_cluster_k5"
].map(
    K5_LABELS
)


write_csv_safe(
    summary_k5,
    TABLE_DIR
    / "DIAGNOSTIC_outlier_summary_by_K5.csv",
)


display(
    summary_k5.round(
        4
    )
)

# Part V — exact PCA influence for strongest candidates

A point can be unusual without materially changing the PCA representation.

For the strongest candidates, this section removes one observation at a time and refits PCA exactly. It records:

- relative change in the first eigenvalue;
- change in cumulative variance explained by the first 10 PCs;
- relative Frobenius change in the covariance matrix;
- principal angles between the full-data and leave-one-out PC10 subspaces.

Large principal-angle or covariance changes indicate a genuine **leverage point** in the representation itself.

In [ ]:
# =============================================================================
# 13. SELECT CANDIDATES FOR EXACT LEAVE-ONE-OUT PCA INFLUENCE
# =============================================================================

# Candidate score based only on global optical diagnostics.
candidate_rank = audit_sorted.copy()


candidate_indices = candidate_rank.index[
    :min(
        INFLUENCE_TOP_N,
        len(
            candidate_rank
        ),
    )
].to_numpy()


print(
    "Exact leave-one-out PCA influence candidates:",
    len(
        candidate_indices
    )
)


display(
    analysis_df.loc[
        candidate_indices,
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "source",
            "review_priority",
            "global_consensus_count",
            "max_abs_z",
            "knn10_mean_distance",
        ],
    ].round(
        4
    )
)

In [ ]:
# =============================================================================
# 14. EXACT LEAVE-ONE-OUT PCA INFLUENCE
# =============================================================================

pca_reference = PCA(
    n_components=INFLUENCE_PCS,
    svd_solver="full",
)


pca_reference.fit(
    X_z
)


V_ref = pca_reference.components_

cov_ref = np.cov(
    X_z,
    rowvar=False,
)


ref_pc1_var = float(
    pca_reference.explained_variance_[
        0
    ]
)

ref_pc10_cum = float(
    pca_reference.explained_variance_ratio_.sum()
)


influence_rows = []


for rank_i, idx in enumerate(
    candidate_indices,
    start=1,
):

    keep = np.ones(
        len(
            X_z
        ),
        dtype=bool,
    )

    keep[
        idx
    ] = False


    pca_loo = PCA(
        n_components=INFLUENCE_PCS,
        svd_solver="full",
    )


    pca_loo.fit(
        X_z[
            keep
        ]
    )


    V_loo = pca_loo.components_


    # Principal angles between the two PC10 subspaces.
    singular_values = np.linalg.svd(
        V_ref
        @ V_loo.T,
        compute_uv=False,
    )


    singular_values = np.clip(
        singular_values,
        -1.0,
        1.0,
    )


    principal_angles_deg = np.degrees(
        np.arccos(
            singular_values
        )
    )


    cov_loo = np.cov(
        X_z[
            keep
        ],
        rowvar=False,
    )


    covariance_relative_change = (
        np.linalg.norm(
            cov_loo
            - cov_ref,
            ord="fro",
        )
        / np.linalg.norm(
            cov_ref,
            ord="fro",
        )
    )


    row = analysis_df.iloc[
        idx
    ]


    influence_rows.append(
        {
            "audit_rank": rank_i,
            "analysis_row_index": int(
                idx
            ),
            "PlotID_model": row[
                "PlotID_model"
            ],
            "_plot_key": row[
                "_plot_key"
            ],
            "veg_year_original": int(
                row[
                    "veg_year_original"
                ]
            ),
            "optical_year": int(
                row[
                    "optical_year"
                ]
            ),
            "source": row[
                "source"
            ],
            "parent_cluster_k5": int(
                row[
                    "parent_cluster_k5"
                ]
            ),
            "parent_cluster_k8": int(
                row[
                    "parent_cluster_k8"
                ]
            ),
            "review_priority": row[
                "review_priority"
            ],
            "global_consensus_count": int(
                row[
                    "global_consensus_count"
                ]
            ),
            "max_abs_z": float(
                row[
                    "max_abs_z"
                ]
            ),
            "knn10_mean_distance": float(
                row[
                    "knn10_mean_distance"
                ]
            ),
            "relative_change_pc1_eigenvalue": float(
                abs(
                    pca_loo.explained_variance_[
                        0
                    ]
                    - ref_pc1_var
                )
                / ref_pc1_var
            ),
            "change_pc10_cumulative_variance": float(
                pca_loo.explained_variance_ratio_.sum()
                - ref_pc10_cum
            ),
            "covariance_relative_frobenius_change": float(
                covariance_relative_change
            ),
            "max_principal_angle_deg_PC10": float(
                principal_angles_deg.max()
            ),
            "mean_principal_angle_deg_PC10": float(
                principal_angles_deg.mean()
            ),
        }
    )


influence_df = pd.DataFrame(
    influence_rows
).sort_values(
    [
        "covariance_relative_frobenius_change",
        "max_principal_angle_deg_PC10",
    ],
    ascending=False,
)


write_csv_safe(
    influence_df,
    TABLE_DIR
    / "DIAGNOSTIC_exact_leave_one_out_PCA_influence.csv",
)


display(
    influence_df.round(
        5
    )
)

# Part VI — feature drivers for flagged observations

For each non-normal observation, the notebook writes the strongest IA45 contributors by absolute annual-z value.

This is often the fastest route from a statistical flag to a technical diagnosis:

> Which harmonic parameter is actually causing the anomaly?

In [ ]:
# =============================================================================
# 15. LONG TABLE OF TOP FEATURE DRIVERS
# =============================================================================

review_mask = analysis_df[
    "review_priority"
].ne(
    "NORMAL"
)


review_indices = np.where(
    review_mask.to_numpy()
)[
    0
]


driver_rows = []


for idx in review_indices:

    row = analysis_df.iloc[
        idx
    ]

    order = np.argsort(
        np.abs(
            X_z[
                idx
            ]
        )
    )[
        ::-1
    ][
        :TOP_FEATURES_PER_CANDIDATE
    ]


    for feature_rank, j in enumerate(
        order,
        start=1,
    ):

        driver_rows.append(
            {
                "analysis_row_index": int(
                    idx
                ),
                "PlotID_model": row[
                    "PlotID_model"
                ],
                "_plot_key": row[
                    "_plot_key"
                ],
                "veg_year_original": int(
                    row[
                        "veg_year_original"
                    ]
                ),
                "optical_year": int(
                    row[
                        "optical_year"
                    ]
                ),
                "source": row[
                    "source"
                ],
                "parent_cluster_k5": int(
                    row[
                        "parent_cluster_k5"
                    ]
                ),
                "parent_cluster_k8": int(
                    row[
                        "parent_cluster_k8"
                    ]
                ),
                "review_priority": row[
                    "review_priority"
                ],
                "global_consensus_count": int(
                    row[
                        "global_consensus_count"
                    ]
                ),
                "feature_rank": feature_rank,
                "feature": FEATURES[
                    j
                ],
                "z_value": float(
                    X_z[
                        idx,
                        j
                    ]
                ),
                "abs_z": float(
                    abs(
                        X_z[
                            idx,
                            j
                        ]
                    )
                ),
            }
        )


driver_df = pd.DataFrame(
    driver_rows
)


write_csv_safe(
    driver_df,
    TABLE_DIR
    / "DIAGNOSTIC_flagged_observation_top_feature_drivers.csv",
)


display(
    driver_df.head(
        50
    ).round(
        4
    )
)

# Part VII — diagnostic figures

In [ ]:
# =============================================================================
# 16. HISTOGRAM — MAX ABSOLUTE Z
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        6,
    )
)


ax.hist(
    analysis_df[
        "max_abs_z"
    ],
    bins=40,
)


for threshold in [
    ABS_Z_REVIEW,
    ABS_Z_HIGH,
    ABS_Z_CRITICAL,
]:

    ax.axvline(
        threshold,
        linestyle="--",
        linewidth=1,
        label=f"|z| = {threshold:g}",
    )


ax.set_xlabel(
    "Maximum absolute annual-z value per observation"
)

ax.set_ylabel(
    "Observations"
)

ax.set_title(
    "Distribution of maximum IA45 optical extremeness"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.2
)


plt.tight_layout()


fig_path = (
    FIG_DIR
    / "histogram_max_abs_z.png"
)


fig.savefig(
    fig_path,
    dpi=250,
    bbox_inches="tight",
)


plt.show()
plt.close(
    fig
)

In [ ]:
# =============================================================================
# 17. GLOBAL EXTREMENESS VS LOCAL ISOLATION
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        7,
    )
)


ax.scatter(
    analysis_df[
        "max_abs_z"
    ],
    analysis_df[
        "knn10_mean_distance"
    ],
    s=18,
    alpha=0.45,
)


label_rows = audit_sorted.head(
    min(
        15,
        len(
            audit_sorted
        ),
    )
)


for idx, row in label_rows.iterrows():

    ax.annotate(
        str(
            row[
                "_plot_key"
            ]
        ),
        (
            row[
                "max_abs_z"
            ],
            row[
                "knn10_mean_distance"
            ],
        ),
        xytext=(
            4,
            4,
        ),
        textcoords="offset points",
        fontsize=8,
    )


ax.set_xlabel(
    "Maximum absolute annual-z"
)

ax.set_ylabel(
    "Mean distance to 10 nearest neighbors in core PCA10"
)

ax.set_title(
    "Global optical extremeness versus local isolation"
)

ax.grid(
    alpha=0.2
)


plt.tight_layout()


fig_path = (
    FIG_DIR
    / "max_abs_z_vs_knn10.png"
)


fig.savefig(
    fig_path,
    dpi=250,
    bbox_inches="tight",
)


plt.show()
plt.close(
    fig
)

In [ ]:
# =============================================================================
# 18. FULL-DATA PCA1 / PCA2 WITH REVIEW CANDIDATES LABELED
# =============================================================================

pca_full_plot = PCA(
    n_components=2,
    svd_solver="full",
)


PC12 = pca_full_plot.fit_transform(
    X_z
)


fig, ax = plt.subplots(
    figsize=(
        10,
        7,
    )
)


ax.scatter(
    PC12[
        :,
        0
    ],
    PC12[
        :,
        1
    ],
    s=18,
    alpha=0.4,
)


for idx, row in audit_sorted.head(
    min(
        15,
        len(
            audit_sorted
        ),
    )
).iterrows():

    ax.annotate(
        str(
            row[
                "_plot_key"
            ]
        ),
        (
            PC12[
                idx,
                0
            ],
            PC12[
                idx,
                1
            ],
        ),
        xytext=(
            4,
            4,
        ),
        textcoords="offset points",
        fontsize=8,
    )


ax.set_xlabel(
    "PC1"
)

ax.set_ylabel(
    "PC2"
)

ax.set_title(
    "IA45 annual-z PCA with strongest outlier candidates labeled"
)

ax.grid(
    alpha=0.2
)


plt.tight_layout()


fig_path = (
    FIG_DIR
    / "PCA12_outlier_candidates.png"
)


fig.savefig(
    fig_path,
    dpi=250,
    bbox_inches="tight",
)


plt.show()
plt.close(
    fig
)

In [ ]:
# =============================================================================
# 19. PCA INFLUENCE FIGURE
# =============================================================================

if len(
    influence_df
) > 0:

    influence_plot = influence_df.head(
        min(
            20,
            len(
                influence_df
            ),
        )
    ).iloc[
        ::-1
    ]


    fig, ax = plt.subplots(
        figsize=(
            10,
            7,
        )
    )


    ax.barh(
        influence_plot[
            "_plot_key"
        ],
        influence_plot[
            "covariance_relative_frobenius_change"
        ],
    )


    ax.set_xlabel(
        "Relative covariance change after removing one observation"
    )

    ax.set_ylabel(
        "Plot"
    )

    ax.set_title(
        "Exact leave-one-out optical covariance influence"
    )

    ax.grid(
        axis="x",
        alpha=0.2
    )


    plt.tight_layout()


    fig_path = (
        FIG_DIR
        / "leave_one_out_covariance_influence.png"
    )


    fig.savefig(
        fig_path,
        dpi=250,
        bbox_inches="tight",
    )


    plt.show()
    plt.close(
        fig
    )

# Part VIII — save the full audit

The final audit table preserves all observations. Nothing is removed.

Recommended interpretation:

- **CRITICAL/HIGH + large PCA influence**: strongest technical-QC candidates;
- **high global isolation but modest class mismatch**: likely globally novel optical observations;
- **normal globally but high same-class distance**: ecological/label/temporal mismatch rather than obvious bad data;
- **one feature responsible for most of the extremeness**: inspect that harmonic parameter and its fit diagnostics first;
- **many features coherently extreme**: potentially a real optical endmember, disturbance, or geolocation/domain issue.

The strongest exclusions should be justified by inspection of the source optical/harmonic data rather than the statistical flag alone.

In [ ]:
# =============================================================================
# 20. SAVE FULL AUDIT + REVIEW SHORTLIST
# =============================================================================

analysis_output = analysis_df.drop(
    columns=[
        "_priority_order",
    ],
    errors="ignore",
).copy()


write_csv_safe(
    analysis_output,
    TABLE_DIR
    / "DIAGNOSTIC_full_IA45_optical_outlier_audit.csv",
)


review_shortlist = audit_sorted.loc[
    audit_sorted[
        "review_priority"
    ].ne(
        "NORMAL"
    )
].drop(
    columns=[
        "_priority_order",
    ],
    errors="ignore",
)


write_csv_safe(
    review_shortlist,
    TABLE_DIR
    / "DIAGNOSTIC_optical_outlier_review_shortlist.csv",
)


print(
    "\nTop review candidates:"
)


display(
    review_shortlist[
        [
            "PlotID_model",
            "_plot_key",
            "veg_year_original",
            "optical_year",
            "source",
            "parent_cluster_k5",
            "parent_cluster_k8",
            "review_priority",
            "global_consensus_count",
            "max_abs_z",
            "rms_z",
            "core_pc10_robust_radius",
            "knn10_mean_distance",
            "global_lw_mahalanobis_sq",
            "k5_sameclass_knn5",
            "k8_sameclass_knn5",
            "most_extreme_feature",
            "most_extreme_feature_z",
        ]
    ].head(
        50
    ).round(
        4
    )
)

In [ ]:
# =============================================================================
# 21. OUTPUT MANIFEST
# =============================================================================

print(
    "=" * 100
)

print(
    "IA45 EXTREME OPTICAL OUTLIER AUDIT COMPLETE"
)

print(
    "=" * 100
)


for label, directory in [
    (
        "Tables",
        TABLE_DIR,
    ),
    (
        "Figures",
        FIG_DIR,
    ),
]:

    print(
        f"\n{label}:"
    )

    for path in sorted(
        directory.glob(
            "*"
        )
    ):

        print(
            " ",
            path.name
        )